# TradeSense · Component 03 · Protocol B2 · step 5: CAL measurement and the final-set declaration (CPU, no training)

Uses only the **saved scores** of notebooks 28 (AE-ctx), 29 (XGBoost v2) and 30 (Transformer v2). Nothing is trained here, no GPU needed. TEST-B / TEST-3 are **never loaded** (only the first 31 days = August 2024 of the Aug–Oct bar file are read, as in notebooks 28–30).

What it does
1. unit tests;
2. **genuine CAL alarm rates** (2024-08-02..08-31): every model, N0/N1, T0/T1, budgets 1/2/4, with the DEV rate next to it and the number of days above 2 × budget (H-B1 evidence);
3. **CAL-3 recall** (200 injected events per symbol, 8 families): T1c (primary), T1 and T0, per family, event-bootstrap 95% intervals, chance baseline;
4. **N0/N1 decisions** (H-B1 and H-B2) for XGBoost v2 and Transformer v2 under the rules of **amendment A3** (below, written before this notebook ran); AE-ctx adoption from notebook 28 stays;
5. **Aug 5 2024** (market crash) descriptive stress check;
6. **union and complementarity** of the three final models;
7. the list of v1 model files on the Drive (for notebook 31b: v1 re-scoring / H-B6 preparation);
8. save everything and write **`declared_final_set_v1.json`**: after this file exists, no model, normalisation, threshold rule or q changes before TEST (notebook 32).

## Amendment A3 (decision rules; also stored in the project doc `protocol_b2_amendment_A3.md`)
# Protocol B2 – Amendment A3 (decision rules of notebook 31) – 2026-10-14
Written BEFORE the CAL measurement of XGBoost v2 and Transformer v2 exists (their CAL scores are saved by notebooks 29/30 but have not been analysed). For AE-ctx the genuine CAL alarm rates were already seen in notebook 28 (its adoption of N1 is frozen, not changed). TEST-B / TEST-3 are untouched.

1. **N0/N1 adoption per model family (XGBoost v2, Transformer v2).** N1 is adopted iff H-B1 AND H-B2, exactly as in the protocol. H-B1: genuine CAL, primary budget 2/day, inflation = mean over BTC/ETH of realised episodes/day divided by 2; N0 with static threshold T0 versus N1 with rolling threshold T1; N1 must cut the excess over the budget by at least 50% and stay within [0.5, 2] x budget. H-B2: VAL-3 recall at the primary budget, macro over all 8 families, mean BTC/ETH, N1 not more than 0.05 below N0. **Rule used for the H-B2 recall: T1c** (amendment A1/A2 reasoning: T1 on the dense synthetic VAL-3 is depressed). T1 (pre-registered) and T0 values are printed next to it; if the T1 reading would give a different adoption this is reported openly, and the T1c reading is the one used.
2. **Deployed threshold rule of every final model: T1** (rolling 7-day own-score quantile, q per budget frozen from the genuine DEV bars). N0/N1 changes only the normalisation. T0 is a drift stress test, never a deployed rule.
3. **CAL-3 recall** is reported under T1c (primary), T1 and T0, at budgets 1/2/4, per family, with event-bootstrap 95% intervals and the chance baseline (genuine alarm series circularly shifted). CAL-3 events lie inside bars 1440..44639 (2024-08-02..08-31); only bars < 44640 of the CAL series are used; thresholds and q come from DEV only.
4. **Union.** The union of the three final models is reported at budgets 1/2/4 per model with the realised genuine union episodes/day; nothing is selected on CAL from the union.
5. **v1 comparison and H-B6.** The v1 model files are not re-scored in notebook 31 (their file formats are listed in the notebook for notebook 31b). H-B6 is decided on TEST-3 in notebook 32 (T1c, T1 and T0 shown side by side; the T1 realised-burden condition as pre-registered).
6. **Aug 5 2024 (market crash day)** is a descriptive stress check (episodes that day versus other CAL days); no decision is attached.
7. After notebook 31 saves its outputs, `declared_final_set_v1.json` is the frozen final set; no model, normalisation, threshold rule or q is changed before TEST.

In [ ]:
import os, json, glob, time, hashlib, warnings, sys, datetime
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'
SYMBOLS = ['BTCUSDT', 'ETHUSDT']; MODES = ['N0', 'N1']
MOCK = os.environ.get('TS_MOCK') == '1'; NBOOT = 200 if MOCK else 1000
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'run notebook 27 first (it freezes the protocol)'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol was changed'
for f in ['ae_ctx_decisions.json', 'xgb_v2_decisions.json', 'transformer_v2_decisions.json', 'b2_step4_share_block.txt']:
    assert (OUT/f).exists(), f'{f} missing: run notebooks 28, 29 (and 29b) and 30 to the end first'
for m in ['ae_ctx', 'xgb_v2', 'transformer_v2']:
    for s in SYMBOLS:
        for md in MODES: assert (OUT/'scores'/f'{m}_{s}_{md}.npz').exists(), f'scores/{m}_{s}_{md}.npz missing'
say('protocol B2 intact, sha256 ' + ph[:16] + ('  [MOCK RUN]' if MOCK else ''))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
protocol B2 intact, sha256 443f99d5faf5929f


## 1 · Library (same file as notebooks 28-30)

In [ ]:
_old_lib = (OUT/'b2_lib.py').read_text() if (OUT/'b2_lib.py').exists() else None
open("b2_lib.py", "w").write(r'''
"""TradeSense C03 - Protocol B2 shared library: normalisation N0/N1, conditional autoencoder (AE-ctx), burden thresholds T0/T1, recall."""
import numpy as np, pandas as pd, torch, torch.nn as nn, time, json, hashlib
from numpy.lib.stride_tricks import sliding_window_view

FEATS = ['log_returns', 'trade_flow_imbalance', 'hl_spread_ratio', 'rush_order_density', 'volume_acceleration', 'large_trade_count']
LOGC = [3, 5]; WIN7 = 10080; MINP = 1440; CTX = 15; WARM = 30; GAPM = 5; CLIP = 20.0
BUDGETS = [1, 2, 4]; PRIMARY = 2; QGRID = [0.99, 0.993, 0.995, 0.997, 0.998, 0.999, 0.9993, 0.9995, 0.9997, 0.9998, 0.9999, 0.99995]
FAMS = {1: 'pump_and_dump_like', 2: 'fake_breakout_like', 3: 'abnormal_volume_burst', 4: 'trade_flow_imbalance', 5: 'wash_like_aggregate_proxy',
        6: 'liquidity_stress_proxy', 7: 'momentum_ignition_like', 8: 'slow_accumulation_like'}


# ---------------------------------------------------------------- normalisation
def to_x(F):
    x = np.asarray(F[FEATS] if hasattr(F, 'columns') else F, dtype='float64').copy()
    for j in LOGC: x[:, j] = np.log1p(x[:, j])
    return x


def fit_n0(F_train):
    """median / (IQR/1.349) on log1p-transformed Layer 1 TRAIN bars (caller passes bars >= 15 only)."""
    x = to_x(F_train); med = np.median(x, axis=0); q75, q25 = np.percentile(x, [75, 25], axis=0); return {'median': med, 'scale': (q75 - q25) / 1.349}


def normalise(F, mode, n0):
    x = to_x(F)
    if mode == 'N0': med, scl = n0['median'][None, :], n0['scale'][None, :]
    else:
        df = pd.DataFrame(x); r = df.rolling(WIN7, min_periods=MINP)
        med = r.median().shift(1).to_numpy(); q75 = r.quantile(0.75).shift(1).to_numpy(); q25 = r.quantile(0.25).shift(1).to_numpy()
        scl = np.maximum((q75 - q25) / 1.349, 0.1 * n0['scale'][None, :])
        bad = np.isnan(med) | np.isnan(scl); med = np.where(bad, n0['median'][None, :], med); scl = np.where(bad, n0['scale'][None, :], scl)
    return np.clip((x - med) / scl, -CLIP, CLIP)


def context(z):
    """causal context: mean and std (ddof 1) of the previous CTX bars; rows < CTX are zero-filled and never used."""
    n = len(z); m = np.zeros_like(z); s = np.zeros_like(z)
    w = sliding_window_view(z, CTX, axis=0)[: n - CTX]            # window k = bars k..k+14 -> bar k+15
    m[CTX:] = w.mean(axis=2); s[CTX:] = w.std(axis=2, ddof=1); return m, s


# ---------------------------------------------------------------- conditional autoencoder
class AECtx(nn.Module):
    def __init__(self, d=6, b=3):
        super().__init__(); self.enc = nn.Sequential(nn.Linear(d + 2 * d, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, b))
        self.dec = nn.Sequential(nn.Linear(b + 2 * d, 32), nn.ReLU(), nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, d))

    def forward(self, z, cm, cs):
        c = torch.cat([cm, cs], 1); return self.dec(torch.cat([self.enc(torch.cat([z, c], 1)), c], 1))


def train_ae(z, idx, seed, max_epochs=150, patience=10, batch=512, log=None):
    """z normalised genuine series; idx = bar indices allowed for fitting (valid, inside Layer 1 TRAIN). 90% fit / 10% early stop, 15-bar gap."""
    torch.manual_seed(seed); rng = np.random.default_rng(seed); cm, cs = context(z)
    idx = np.sort(idx); cut = int(0.9 * len(idx)); fit = idx[:cut]; val = idx[idx > idx[cut] + CTX]
    T = lambda a: torch.tensor(a, dtype=torch.float32)
    Zf, Mf, Sf = T(z[fit]), T(cm[fit]), T(cs[fit]); Zv, Mv, Sv = T(z[val]), T(cm[val]), T(cs[val])
    net = AECtx(); opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5); best, bad, state = 1e18, 0, None
    for ep in range(max_epochs):
        net.train(); perm = torch.tensor(rng.permutation(len(fit)))
        for i in range(0, len(perm), batch):
            b = perm[i:i + batch]; opt.zero_grad(); loss = ((net(Zf[b], Mf[b], Sf[b]) - Zf[b]) ** 2).mean(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad(): v = float(((net(Zv, Mv, Sv) - Zv) ** 2).mean())
        if v < best - 1e-6: best, bad, state = v, 0, {k: x.clone() for k, x in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(state); net.eval(); return net, {'epochs': ep + 1, 'best_val': best, 'fit_rows': int(len(fit)), 'val_rows': int(len(val)), 'fit_min': int(fit.min()), 'fit_max': int(fit.max())}


def ae_errors(net, z):
    cm, cs = context(z); T = lambda a: torch.tensor(a, dtype=torch.float32)
    with torch.no_grad(): rec = net(T(z), T(cm), T(cs)).numpy()
    return (z - rec) ** 2


def p99_scale(err, idx): return np.percentile(err[idx], 99, axis=0) + 1e-12


def ae_columns(err, scale):
    """8 columns: fmax score, fmean score, 6 per-feature errors / TRAIN p99 scale."""
    r = err / scale[None, :]; return np.column_stack([r.max(axis=1), err.mean(axis=1), r])


# ---------------------------------------------------------------- burden: episodes, thresholds, recall
def episodes(alarm, gap=GAPM):
    a = np.flatnonzero(alarm)
    if len(a) == 0: return 0
    return int(1 + np.sum(np.diff(a) > gap))


def eps_per_day(alarm, valid):
    a = alarm & valid; days = valid.sum() / 1440.0
    return episodes(a) / max(days, 1e-9)


def valid_mask(n, lo, hi): m = np.zeros(n, bool); m[max(lo, WARM):hi] = True; return m


def t0_threshold(score, valid, budget):
    """lowest quantile-grid threshold (descending probability grid, lowering the threshold step by step) with episodes/day <= budget."""
    s = score[valid]; best = None
    for q in sorted(QGRID, reverse=True):
        thr = np.quantile(s, q)
        if eps_per_day(score > thr, valid) <= budget: best = thr
        else: break
    return best if best is not None else np.quantile(s, max(QGRID))


def rolling_thr(score, q):
    return pd.Series(score).rolling(WIN7, min_periods=MINP).quantile(q).shift(1).to_numpy()


def t1_q(score, valid, budget):
    """smallest quantile level q (lowest rolling threshold) with episodes/day <= budget on the genuine DEV bars."""
    for q in QGRID:
        thr = rolling_thr(score, q); alarm = (score > np.where(np.isnan(thr), np.inf, thr))
        if eps_per_day(alarm, valid) <= budget: return q
    return QGRID[-1]


def alarms(score, rule, par):
    if rule == 'T0': return score > par
    thr = rolling_thr(score, par); return score > np.where(np.isnan(thr), np.inf, thr)


def recall_by_family(alarm, ev, seg_ok=None):
    """ev rows: t0, dur, family, (replica); detected if an alarm falls in [t0, t0+dur-1]."""
    c = np.concatenate([[0], np.cumsum(alarm.astype(np.int64))]); out = {}
    for fam, sub in ev.groupby('family'):
        hit = [(c[r.t0 + r.dur] - c[r.t0]) > 0 for r in sub.itertuples()]; out[int(fam)] = (int(np.sum(hit)), int(len(hit)))
    return out


def bootstrap_ci(hits, n=1000, seed=0):
    rng = np.random.default_rng(seed); h = np.asarray(hits, float)
    if len(h) == 0: return (np.nan, np.nan)
    b = [h[rng.integers(0, len(h), len(h))].mean() for _ in range(n)]; return (float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5)))


def macro(rec, fams):
    v = [rec[f][0] / rec[f][1] for f in fams if f in rec and rec[f][1] > 0]; return float(np.mean(v)) if v else float('nan')


def sha_bytes(b): return hashlib.sha256(b).hexdigest()

''')

import sys, importlib; sys.path.insert(0, '.'); import b2_lib as B; importlib.reload(B); (OUT/'b2_lib.py').write_text(Path('b2_lib.py').read_text())
say('library sha256 ' + sha_file('b2_lib.py'))
assert _old_lib is None or _old_lib.strip() == Path('b2_lib.py').read_text().strip(), 'b2_lib.py differs from the copy written by notebook 28'

library sha256 1b0b46630763c944


## 2 · Data (saved scores, events) and threshold helpers

In [ ]:
AOLEN = 31 * 1440; CAL_LO = 1440; PRIM = B.PRIMARY; BUD = B.BUDGETS
L1V = B.valid_mask(129600, 90720, 129600); CALV = B.valid_mask(AOLEN, CAL_LO, AOLEN)
MODELS = ['AE-ctx fmax', 'XGBoost v2', 'Transformer v2']; FILES = {'AE-ctx fmax': 'ae_ctx', 'XGBoost v2': 'xgb_v2', 'Transformer v2': 'transformer_v2'}
OLD, NEW = [1, 2, 3, 4, 5, 6], [7, 8]; ALLF = OLD + NEW
def nanw(a): a = np.asarray(a, 'float64').copy(); a[:B.WARM] = np.nan; return a
SCR = {}; EV = {}; PRICE = {}; TIDX = {}
for s in SYMBOLS:
    ao = pd.read_parquet(BASE/'layer1_aug_oct_v1'/f'{s.lower()}_l1_feature_matrix_aug_oct_2024.parquet', columns=['close']).iloc[:AOLEN]   # August only: TEST-B is never used
    assert len(ao) == AOLEN and str(ao.index[0])[:10] == '2024-08-01' and str(ao.index[-1])[:10] == '2024-08-31', (s, len(ao), ao.index[0], ao.index[-1])
    TIDX[s] = ao.index; PRICE[s] = ao['close'].to_numpy('float64')
    for name in ['VAL', 'CAL']: EV[(s, name)] = pd.read_csv(L3/f'l3v3_{s}_{name}_events.csv')
    for model, fn in FILES.items():
        for mode in MODES:
            z = np.load(OUT/'scores'/f'{fn}_{s}_{mode}.npz'); assert not any('TEST' in k for k in z.files)
            d = {}
            for k in z.files:
                if k == 'scale': continue
                a = z[k]; a = a[:, 0] if a.ndim == 2 else a                  # AE-ctx: column 0 = fmax
                if k.startswith('CAL'): a = a[:AOLEN]
                d[k] = nanw(a)
            assert len(d['GEN_L1']) == 129600 and len(d['GEN_AO']) == AOLEN and len(d['CAL_r0']) == AOLEN, (model, s, mode)
            SCR[(model, s, mode)] = d
def event_sets(name):
    out = []
    for s in SYMBOLS:
        ev = EV[(s, name)]
        for r in sorted(ev.replica.unique()): out.append((s, f'{name}_r{int(r)}', ev[ev.replica == r].reset_index(drop=True)))
    return out
ESETS = {'VAL': event_sets('VAL'), 'CAL': event_sets('CAL')}
META = {n: pd.concat([pd.DataFrame({'sym': s, 'fam': ev.family.to_numpy()}) for s, k, ev in ESETS[n]], ignore_index=True) for n in ESETS}
for s, k, ev in ESETS['CAL']: assert ev.t0.min() >= CAL_LO and (ev.t0 + ev.dur).max() <= AOLEN, 'CAL-3 events must lie inside 2024-08-02..08-31'
for s, k, ev in ESETS['VAL']: assert ev.t0.min() >= 90720 and (ev.t0 + ev.dur).max() <= 129600
say('events: ' + str({n: META[n].groupby('sym').size().to_dict() for n in META}) + '; CAL-3 per family (both symbols): ' + str(META['CAL'].fam.value_counts().sort_index().to_dict()))
# ---------- thresholds and alarms (parameters from the genuine DEV bars only)
RT = {}; PAR = {}; HC = {}
def rthr(model, s, mode, key, q):
    k = (model, s, mode, key, q)
    if k not in RT: t = B.rolling_thr(SCR[(model, s, mode)][key], q); RT[k] = np.where(np.isnan(t), np.inf, t)
    return RT[k]
def par(model, s, mode, bud):
    k = (model, s, mode, bud)
    if k not in PAR:
        g = SCR[(model, s, mode)]['GEN_L1']
        q = next((q for q in B.QGRID if B.eps_per_day(g > rthr(model, s, mode, 'GEN_L1', q), L1V) <= bud), B.QGRID[-1])
        PAR[k] = {'T0': float(B.t0_threshold(g, L1V, bud)), 'T1': q}
    return PAR[k]
def alarm(model, s, mode, key, rule, bud, ref=None):
    p = par(model, s, mode, bud); x = SCR[(model, s, mode)][key]
    if rule == 'T0': return x > p['T0']
    if rule == 'T1': return x > rthr(model, s, mode, key, p['T1'])
    return x > rthr(model, s, mode, ref, p['T1'])                 # T1c: rolling threshold of the genuine series applied to the injected series
def event_hits(a, ev):
    c = np.concatenate([[0], np.cumsum(a.astype(np.int64))]); t0 = ev.t0.to_numpy(); du = ev.dur.to_numpy(); return (c[t0 + du] - c[t0]) > 0
def hits(model, mode, name, rule, bud):
    k = (model, mode, name, rule, bud)
    if k not in HC:
        ref = 'GEN_L1' if name == 'VAL' else 'GEN_AO'
        HC[k] = np.concatenate([event_hits(alarm(model, s, mode, key, rule, bud, ref=ref), ev) for s, key, ev in ESETS[name]])
    return HC[k]
def gen_alarm(model, s, mode, rule, bud, where):
    key = 'GEN_L1' if where == 'DEV' else 'GEN_AO'; return alarm(model, s, mode, key, 'T0' if rule == 'T0' else 'T1', bud)
def gen_eps(model, mode, rule, bud, where):
    valid = L1V if where == 'DEV' else CALV; return float(np.mean([B.eps_per_day(gen_alarm(model, s, mode, rule, bud, where), valid) for s in SYMBOLS]))
def macro_of(h, name, fams):
    m = META[name]; v = []
    for s in SYMBOLS:
        sel = ((m.sym == s) & m.fam.isin(fams)).to_numpy(); d = pd.DataFrame({'fam': m.fam[sel].to_numpy(), 'h': h[sel]}); v.append(d.groupby('fam').h.mean().mean())
    return float(np.mean(v))
def fam_rates(h, name):
    m = META[name]; return pd.DataFrame({'sym': m.sym, 'fam': m.fam, 'h': h}).groupby(['sym', 'fam']).h.mean().groupby('fam').mean()
def macro_boot(h, name, fams, n=None, seed=0):
    n = n or NBOOT; m = META[name]; groups = []
    for s in SYMBOLS:
        gs = []
        for f in fams:
            idx = np.flatnonzero(((m.sym == s) & (m.fam == f)).to_numpy())
            if len(idx): gs.append(h[idx].astype(float))
        groups.append(gs)
    rg = np.random.default_rng(seed); out = [np.mean([np.mean([g[rg.integers(0, len(g), len(g))].mean() for g in gs]) for gs in groups]) for _ in range(n)]
    return tuple(float(x) for x in np.percentile(out, [2.5, 97.5]))
def chance_macro(alarm_by_sym, fams, nsh=40, seed=0):
    rg = np.random.default_rng(seed); out = []
    for _ in range(nsh):
        hs = []
        for s, key, ev in ESETS['CAL']:
            seg = alarm_by_sym[s][CAL_LO:AOLEN]; full = np.zeros(AOLEN, bool); full[CAL_LO:] = np.roll(seg, int(rg.integers(0, len(seg)))); hs.append(event_hits(full, ev))
        out.append(macro_of(np.concatenate(hs), 'CAL', fams))
    return float(np.mean(out))
def ep_starts(a):
    i = np.flatnonzero(a)
    if len(i) == 0: return np.array([], int)
    return i[np.concatenate([[True], np.diff(i) > B.GAPM])]
say('data ready: models ' + ', '.join(MODELS) + '; scores sliced to bars < 44640 (August) for CAL; parameters (T0 threshold, q) from the genuine DEV bars only')

events: {'VAL': {'BTCUSDT': 300, 'ETHUSDT': 300}, 'CAL': {'BTCUSDT': 200, 'ETHUSDT': 200}}; CAL-3 per family (both symbols): {1: 50, 2: 50, 3: 50, 4: 50, 5: 50, 6: 50, 7: 50, 8: 50}
data ready: models AE-ctx fmax, XGBoost v2, Transformer v2; scores sliced to bars < 44640 (August) for CAL; parameters (T0 threshold, q) from the genuine DEV bars only


## 3 · Unit tests

In [ ]:
# T01 toy episode counting
a = np.zeros(100, bool); a[[10, 11, 12, 30]] = True; assert B.episodes(a) == 2; a2 = np.zeros(100, bool); a2[[10, 14]] = True; assert B.episodes(a2) == 1; say('PASS T01 episode counting (gap rule)')
# T02 recall: vectorised == plain loop on random alarms
rg = np.random.default_rng(0); s0, k0, ev0 = ESETS['CAL'][0]
for _ in range(5):
    al = rg.random(AOLEN) < 0.002; fast = event_hits(al, ev0); slow = np.array([al[r.t0:r.t0 + r.dur].any() for r in ev0.itertuples()]); assert (fast == slow).all()
say('PASS T02 event hit rule equals a plain loop')
# T03 cached q equals the library t1_q
mm = ('XGBoost v2', 'BTCUSDT', 'N0'); g = SCR[mm]['GEN_L1']
for b_ in BUD: assert par(*mm, b_)['T1'] == B.t1_q(g, L1V, b_), b_
assert abs(par(*mm, 2)['T0'] - B.t0_threshold(g, L1V, 2)) < 1e-12; say('PASS T03 q and T0 threshold equal the library functions')
# T04 causality of the rolling threshold
x = SCR[mm]['GEN_AO'].copy(); t_ = 20000; y = x.copy(); y[t_:] = np.nanmax(x) * 5
assert np.allclose(np.nan_to_num(B.rolling_thr(x, 0.998)[:t_]), np.nan_to_num(B.rolling_thr(y, 0.998)[:t_])); say('PASS T04 rolling threshold at bar t does not depend on scores at or after t')
# T05 parameters depend on DEV genuine scores only
save = SCR[mm]['GEN_AO']; p_before = dict(par(*mm, 2)); SCR[mm]['GEN_AO'] = np.zeros_like(save); PAR.pop((*mm, 2)); assert par(*mm, 2) == p_before; SCR[mm]['GEN_AO'] = save; say('PASS T05 T0 threshold and q do not change when the CAL genuine scores change')
# T06 only August bars are used
assert all(len(SCR[(m_, s, md)]['CAL_r0']) == AOLEN for m_ in MODELS for s in SYMBOLS for md in MODES)
assert CALV[:CAL_LO].sum() == 0 and CALV.sum() <= AOLEN - CAL_LO; say('PASS T06 CAL window = bars 1440..44639 only; series sliced to 44640 bars')
# T07 T1c equals T1 on the genuine series
a_ = alarm(*mm, 'GEN_AO', 'T1', 2); b_ = alarm(*mm, 'GEN_AO', 'T1c', 2, ref='GEN_AO'); assert (a_ == b_).all(); say('PASS T07 T1c = T1 on genuine bars')
# T08 DEV burden respects the budget
for model in MODELS:
    for mode in MODES:
        for b_ in BUD: assert gen_eps(model, mode, 'T1', b_, 'DEV') <= b_ + 1e-9 and gen_eps(model, mode, 'T0', b_, 'DEV') <= b_ + 1e-9, (model, mode, b_)
say('PASS T08 DEV genuine episodes/day <= budget for every model, normalisation, rule')

PASS T01 episode counting (gap rule)
PASS T02 event hit rule equals a plain loop
PASS T03 q and T0 threshold equal the library functions
PASS T04 rolling threshold at bar t does not depend on scores at or after t
PASS T05 T0 threshold and q do not change when the CAL genuine scores change
PASS T06 CAL window = bars 1440..44639 only; series sliced to 44640 bars
PASS T07 T1c = T1 on genuine bars
PASS T08 DEV genuine episodes/day <= budget for every model, normalisation, rule


## 4 · Genuine CAL alarm rates (H-B1 evidence)

In [ ]:
rows = []
for model in MODELS:
    for mode in MODES:
        for rule in ['T0', 'T1']:
            for b_ in BUD:
                rows.append({'model': model, 'norm': mode, 'thr': rule, 'budget': b_, 'DEV eps/day': gen_eps(model, mode, rule, b_, 'DEV'), 'CAL eps/day': gen_eps(model, mode, rule, b_, 'CAL')})
RATES = pd.DataFrame(rows); RATES['CAL / budget'] = RATES['CAL eps/day'] / RATES['budget']
say('--- genuine alarm episodes per day: DEV (where the rule was fixed) versus CAL (2024-08-02..08-31) ---'); say(RATES.round(2).to_string(index=False))
def daily_eps(a, valid):
    return np.array([B.episodes(a[d * 1440:(d + 1) * 1440] & valid[d * 1440:(d + 1) * 1440]) for d in range(1, AOLEN // 1440)])
drows = []
for model in MODELS:
    for mode in MODES:
        for rule in ['T0', 'T1']:
            de = np.mean([daily_eps(gen_alarm(model, s, mode, rule, PRIM, 'CAL'), CALV) for s in SYMBOLS], axis=0)
            drows.append({'model': model, 'norm': mode, 'thr': rule, 'median eps/day': float(np.median(de)), 'max day': float(de.max()), 'days > 2 x budget': int((de > 2 * PRIM).sum()), 'days': len(de)})
DAYS = pd.DataFrame(drows); say(''); say('--- distribution over the 30 CAL days (mean of BTC/ETH), budget 2/day ---'); say(DAYS.round(2).to_string(index=False))

--- genuine alarm episodes per day: DEV (where the rule was fixed) versus CAL (2024-08-02..08-31) ---
         model norm thr  budget  DEV eps/day  CAL eps/day  CAL / budget
   AE-ctx fmax   N0  T0       1         0.94         0.88          0.88
   AE-ctx fmax   N0  T0       2         1.30         1.28          0.64
   AE-ctx fmax   N0  T0       4         3.85         4.02          1.00
   AE-ctx fmax   N0  T1       1         0.81         1.15          1.15
   AE-ctx fmax   N0  T1       2         1.39         1.68          0.84
   AE-ctx fmax   N0  T1       4         3.17         3.68          0.92
   AE-ctx fmax   N1  T0       1         0.81         0.45          0.45
   AE-ctx fmax   N1  T0       2         1.28         0.72          0.36
   AE-ctx fmax   N1  T0       4         3.81         3.13          0.78
   AE-ctx fmax   N1  T1       1         0.87         0.92          0.92
   AE-ctx fmax   N1  T1       2         1.43         1.43          0.72
   AE-ctx fmax   N1  T1       4   

## 5 · CAL-3 recall (T1c / T1 / T0), per family, chance

In [ ]:
rows = []; FAMTAB = []
for model in MODELS:
    for mode in MODES:
        for rule in ['T1c', 'T1', 'T0']:
            ra = 'T0' if rule == 'T0' else 'T1'
            for b_ in BUD:
                h = hits(model, mode, 'CAL', rule, b_); ch_a = {s: gen_alarm(model, s, mode, ra, b_, 'CAL') for s in SYMBOLS}
                o, n_, a8 = macro_of(h, 'CAL', OLD), macro_of(h, 'CAL', NEW), macro_of(h, 'CAL', ALLF); co, cn = macro_boot(h, 'CAL', OLD), macro_boot(h, 'CAL', NEW)
                rows.append({'model': model, 'norm': mode, 'rule': rule, 'budget': b_, 'CAL genuine eps/day': gen_eps(model, mode, ra, b_, 'CAL'), 'recall 6 old': o, 'old 95% CI': f'{co[0]:.2f}-{co[1]:.2f}', 'recall 2 new': n_, 'new 95% CI': f'{cn[0]:.2f}-{cn[1]:.2f}', 'macro 8': a8,
                             'chance old': chance_macro(ch_a, OLD) if b_ == PRIM else np.nan, 'chance new': chance_macro(ch_a, NEW) if b_ == PRIM else np.nan})
                if b_ == PRIM:
                    fr = fam_rates(h, 'CAL'); FAMTAB.append({'model': model, 'norm': mode, 'rule': rule, **{B.FAMS[f][:14]: float(fr.get(f, np.nan)) for f in ALLF}})
CAL3 = pd.DataFrame(rows); FAMT = pd.DataFrame(FAMTAB)
say('--- CAL-3 recall (200 events per symbol, 8 families), mean of BTC/ETH; chance = genuine alarm series circularly shifted ---'); say(CAL3[CAL3.budget == PRIM].round(3).to_string(index=False))
say(''); say('--- CAL-3 per family, budget 2/day ---'); say(FAMT.round(2).to_string(index=False))
say(''); say('--- CAL-3 recall at budgets 1 / 4 (T1c) ---'); say(CAL3[(CAL3.budget != PRIM) & (CAL3.rule == 'T1c')][['model', 'norm', 'budget', 'CAL genuine eps/day', 'recall 6 old', 'recall 2 new', 'macro 8']].round(3).to_string(index=False))

--- CAL-3 recall (200 events per symbol, 8 families), mean of BTC/ETH; chance = genuine alarm series circularly shifted ---
         model norm rule  budget  CAL genuine eps/day  recall 6 old old 95% CI  recall 2 new new 95% CI  macro 8  chance old  chance new
   AE-ctx fmax   N0  T1c       2                1.683         0.233  0.20-0.27          0.11  0.05-0.17    0.202       0.022       0.056
   AE-ctx fmax   N0   T1       2                1.683         0.150  0.12-0.18          0.01  0.00-0.03    0.115       0.022       0.056
   AE-ctx fmax   N0   T0       2                1.283         0.203  0.17-0.24          0.07  0.03-0.12    0.170       0.016       0.042
   AE-ctx fmax   N1  T1c       2                1.433         0.220  0.18-0.26          0.16  0.10-0.24    0.205       0.020       0.047
   AE-ctx fmax   N1   T1       2                1.433         0.140  0.11-0.17          0.04  0.01-0.08    0.115       0.020       0.047
   AE-ctx fmax   N1   T0       2                0.717 

## 6 · N0 / N1 decisions (H-B1, H-B2, amendment A3)

In [ ]:
AED = json.load(open(OUT/'ae_ctx_decisions.json')); XD = json.load(open(OUT/'xgb_v2_decisions.json')); TD = json.load(open(OUT/'transformer_v2_decisions.json'))
DEC = {}; FINALN = {}
def val8(model, mode, rule): return macro_of(hits(model, mode, 'VAL', rule, PRIM), 'VAL', ALLF)
say('--- N0 / N1 decisions (amendment A3) ---')
for model in MODELS:
    i0 = gen_eps(model, 'N0', 'T0', PRIM, 'CAL') / PRIM; i1 = gen_eps(model, 'N1', 'T1', PRIM, 'CAL') / PRIM
    ex0, ex1 = max(i0 - 1, 0), max(i1 - 1, 0); prem = i0 > 2.0; hb1 = (ex1 <= 0.5 * ex0) and (0.5 <= i1 <= 2.0)
    v = {r: (val8(model, 'N0', r), val8(model, 'N1', r)) for r in ['T1c', 'T1', 'T0']}; hb2 = {r: v[r][1] >= v[r][0] - 0.05 for r in v}
    adopt = bool(hb1 and hb2['T1c']); alt = bool(hb1 and hb2['T1'])
    DEC[model] = {'inflation_N0_T0': i0, 'inflation_N1_T1': i1, 'H-B1 premise (N0/T0 > 2x budget)': bool(prem), 'H-B1': bool(hb1), 'VAL-3 macro8 (N0, N1)': {r: [float(v[r][0]), float(v[r][1])] for r in v},
                  'H-B2 (T1c, decision rule)': bool(hb2['T1c']), 'H-B2 (T1, pre-registered reading)': bool(hb2['T1']), 'H-B2 (T0)': bool(hb2['T0']), 'adopt N1 (A3 rule)': adopt, 'adopt N1 if T1 were used for H-B2': alt}
    if model == 'AE-ctx fmax':
        frozen = bool(AED['AE-ctx fmax']['adopt N1']); DEC[model]['frozen decision from notebook 28'] = frozen; DEC[model]['A3 reading agrees with the frozen one'] = (adopt == frozen); FINALN[model] = 'N1' if frozen else 'N0'
    else: FINALN[model] = 'N1' if adopt else 'N0'
    say(f'{model}: CAL inflation (x budget) N0/T0 {i0:.2f} -> N1/T1 {i1:.2f}; H-B1 premise {prem}, H-B1 {hb1}; VAL-3 macro8 N0 vs N1: T1c {v["T1c"][0]:.3f}/{v["T1c"][1]:.3f} (H-B2 {hb2["T1c"]}), T1 {v["T1"][0]:.3f}/{v["T1"][1]:.3f} ({hb2["T1"]}), T0 {v["T0"][0]:.3f}/{v["T0"][1]:.3f} ({hb2["T0"]}); final: {FINALN[model]}' + ('' if adopt == alt else '   <<< T1 reading would decide differently (reported openly; the T1c reading is used, amendment A3) >>>'))
    if model == 'AE-ctx fmax': say(f'   AE-ctx: frozen notebook-28 decision {DEC[model]["frozen decision from notebook 28"]}; this notebook (A3 rule) says {adopt}; the frozen decision stands')
say('FINAL normalisation per model: ' + str(FINALN))
say('H-B1 summary: premise (N0/T0 alarm rate > 2x budget on CAL) true for ' + str([m_ for m_ in MODELS if DEC[m_]['H-B1 premise (N0/T0 > 2x budget)']]) + '; H-B1 true for ' + str([m_ for m_ in MODELS if DEC[m_]['H-B1']]))
FINAL_TAB = CAL3[(CAL3.budget == PRIM) & CAL3.apply(lambda r: FINALN[r.model] == r.norm, axis=1)][['model', 'norm', 'rule', 'CAL genuine eps/day', 'recall 6 old', 'old 95% CI', 'recall 2 new', 'new 95% CI', 'macro 8', 'chance old', 'chance new']]
say(''); say('--- final models on CAL-3 (budget 2/day) ---'); say(FINAL_TAB.round(3).to_string(index=False))

--- N0 / N1 decisions (amendment A3) ---
AE-ctx fmax: CAL inflation (x budget) N0/T0 0.64 -> N1/T1 0.72; H-B1 premise False, H-B1 True; VAL-3 macro8 N0 vs N1: T1c 0.193/0.158 (H-B2 True), T1 0.129/0.118 (True), T0 0.191/0.153 (True); final: N1
   AE-ctx: frozen notebook-28 decision True; this notebook (A3 rule) says True; the frozen decision stands
XGBoost v2: CAL inflation (x budget) N0/T0 1.14 -> N1/T1 0.72; H-B1 premise False, H-B1 True; VAL-3 macro8 N0 vs N1: T1c 0.719/0.622 (H-B2 False), T1 0.114/0.121 (True), T0 0.714/0.628 (False); final: N0   <<< T1 reading would decide differently (reported openly; the T1c reading is used, amendment A3) >>>
Transformer v2: CAL inflation (x budget) N0/T0 1.19 -> N1/T1 0.88; H-B1 premise False, H-B1 True; VAL-3 macro8 N0 vs N1: T1c 0.734/0.685 (H-B2 True), T1 0.106/0.110 (True), T0 0.714/0.683 (True); final: N1
FINAL normalisation per model: {'AE-ctx fmax': 'N1', 'XGBoost v2': 'N0', 'Transformer v2': 'N1'}
H-B1 summary: premise (N0/T0 alarm rate

## 7 · Aug 5 2024 stress check

In [ ]:
D5 = 4; lo, hi = D5 * 1440, (D5 + 1) * 1440; crows = []; elist = []
for s in SYMBOLS:
    p = PRICE[s]; day = p[lo:hi]; r60 = np.log(day[60:] / day[:-60]); k = int(np.argmin(r60)); say(f'{s} 2024-08-05: close {day[0]:.1f} -> {day[-1]:.1f}, low {day.min():.1f}, day range {100 * (day.max() / day.min() - 1):.1f}%, worst 60-min return {100 * (np.exp(r60[k]) - 1):.1f}% ending {TIDX[s][lo + k + 60]}')
for model in MODELS:
    mode = FINALN[model]
    for rule in ['T1', 'T0']:
        per_sym = {s: daily_eps(gen_alarm(model, s, mode, rule, PRIM, 'CAL'), CALV) for s in SYMBOLS}; de = np.mean([per_sym[s] for s in SYMBOLS], axis=0)   # index 0 = Aug 2 -> Aug 5 = index 3
        a5 = float(de[D5 - 1]); other = np.delete(de, D5 - 1); rank = int((de > a5).sum() + 1)
        crows.append({'model': model, 'norm': mode, 'rule': rule, 'Aug 5 episodes (mean BTC/ETH)': a5, 'median other days': float(np.median(other)), 'max other days': float(other.max()), 'rank of Aug 5 (1 = most)': rank, 'of days': len(de)})
        if rule == 'T1':
            for s in SYMBOLS:
                st = ep_starts(gen_alarm(model, s, mode, rule, PRIM, 'CAL')[lo:hi]); elist.append(f'{model} {s}: ' + ', '.join(str(TIDX[s][lo + i])[11:16] for i in st))
CRASHT = pd.DataFrame(crows); say(''); say('--- Aug 5 2024 stress check (final normalisation, budget 2/day) ---'); say(CRASHT.round(2).to_string(index=False))
say('episode start times (UTC) on Aug 5 under T1:'); [say('  ' + e) for e in elist]

BTCUSDT 2024-08-05: close 58208.0 -> 54018.8, low 49338.5, day range 18.2%, worst 60-min return -8.1% ending 2024-08-05 01:10:00+00:00
ETHUSDT 2024-08-05: close 2693.0 -> 2419.6, low 2140.6, day range 26.0%, worst 60-min return -19.8% ending 2024-08-05 01:10:00+00:00

--- Aug 5 2024 stress check (final normalisation, budget 2/day) ---
         model norm rule  Aug 5 episodes (mean BTC/ETH)  median other days  max other days  rank of Aug 5 (1 = most)  of days
   AE-ctx fmax   N1   T1                            1.5                1.0             3.5                        11       30
   AE-ctx fmax   N1   T0                            0.5                0.5             3.0                        14       30
    XGBoost v2   N0   T1                            1.5                1.0             4.0                         5       30
    XGBoost v2   N0   T0                            6.0                2.0             6.0                         1       30
Transformer v2   N1   T1         

[None, None, None, None, None, None]

## 8 · Union and complementarity of the three final models

In [ ]:
urows = []; fm = FINALN
for rule in ['T1c', 'T1', 'T0']:
    ra = 'T0' if rule == 'T0' else 'T1'
    for b_ in BUD:
        hs = [hits(m_, fm[m_], 'CAL', rule, b_) for m_ in MODELS]; h = np.any(hs, axis=0)
        ge = float(np.mean([B.eps_per_day(np.any([gen_alarm(m_, s, fm[m_], ra, b_, 'CAL') for m_ in MODELS], axis=0), CALV) for s in SYMBOLS]))
        gd = float(np.mean([B.eps_per_day(np.any([gen_alarm(m_, s, fm[m_], ra, b_, 'DEV') for m_ in MODELS], axis=0), L1V) for s in SYMBOLS]))
        co, cn = macro_boot(h, 'CAL', OLD), macro_boot(h, 'CAL', NEW)
        urows.append({'rule': rule, 'budget each': b_, 'union genuine eps/day DEV': gd, 'union genuine eps/day CAL': ge, 'union recall 6 old': macro_of(h, 'CAL', OLD), 'old 95% CI': f'{co[0]:.2f}-{co[1]:.2f}', 'union recall 2 new': macro_of(h, 'CAL', NEW), 'new 95% CI': f'{cn[0]:.2f}-{cn[1]:.2f}', 'best single old': max(macro_of(x_, 'CAL', OLD) for x_ in hs), 'best single new': max(macro_of(x_, 'CAL', NEW) for x_ in hs)})
        assert macro_of(h, 'CAL', ALLF) >= max(macro_of(x_, 'CAL', ALLF) for x_ in hs) - 1e-12      # T09 union is never below a single model
UNI = pd.DataFrame(urows); say('PASS T09 union recall >= best single model'); say('--- union of the three final models on CAL-3 (each model at its own budget; the union burden is measured, not assumed) ---'); say(UNI.round(3).to_string(index=False))
hs = {m_: hits(m_, fm[m_], 'CAL', 'T1c', PRIM) for m_ in MODELS}; M_ = META['CAL']; comp = []
for f in ALLF:
    sel = (M_.fam == f).to_numpy(); n_ = int(sel.sum()); H = np.array([hs[m_][sel] for m_ in MODELS]); k = H.sum(axis=0)
    row = {'family': B.FAMS[f], 'events': n_, 'none': float((k == 0).mean()), '>=2 models': float((k >= 2).mean())}
    for i, m_ in enumerate(MODELS): row[m_ + ' only'] = float(((k == 1) & H[i]).mean())
    comp.append(row)
COMP = pd.DataFrame(comp); say(''); say('--- who finds what (CAL-3, T1c, 2/day each; share of events) ---'); say(COMP.round(2).to_string(index=False))

PASS T09 union recall >= best single model
--- union of the three final models on CAL-3 (each model at its own budget; the union burden is measured, not assumed) ---
rule  budget each  union genuine eps/day DEV  union genuine eps/day CAL  union recall 6 old old 95% CI  union recall 2 new new 95% CI  best single old  best single new
 T1c            1                      2.185                      2.500               0.783  0.75-0.82                0.82  0.75-0.89            0.690             0.72
 T1c            2                      4.130                      3.950               0.817  0.78-0.85                0.85  0.78-0.92            0.730             0.75
 T1c            4                      8.352                      7.883               0.853  0.82-0.88                0.87  0.80-0.93            0.760             0.81
  T1            1                      2.185                      2.500               0.227  0.19-0.27                0.04  0.01-0.08            0.110            

## 9 · v1 files on the Drive (for notebook 31b)

In [ ]:
say('--- v1 model files found on the Drive (for notebook 31b: v1 re-scoring and H-B6; not used in this notebook) ---')
for pat in ['*xgboost_v1*', '*transformer_v1*', '*autoencoder_v1*']:
    for d in sorted(glob.glob(str(BASE/'**'/pat), recursive=True)):
        if 'backup' in d.lower(): continue
        p = Path(d)
        if p.is_dir():
            fl = sorted(x for x in p.rglob('*') if x.is_file()); say(f'{p.relative_to(BASE)}: {len(fl)} files'); [say('    ' + str(x.relative_to(p)) + f'  ({x.stat().st_size // 1024} KB)') for x in fl[:25]]
        else: say(f'{p.relative_to(BASE)} (file, {p.stat().st_size // 1024} KB)')

--- v1 model files found on the Drive (for notebook 31b: v1 re-scoring and H-B6; not used in this notebook) ---
models_xgboost_v1: 22 files
    README.md  (0 KB)
    manifest.json  (3 KB)
    models/pooled/seed11.json  (898 KB)
    models/pooled/seed22.json  (757 KB)
    models/pooled/seed33.json  (974 KB)
    models/selected_config.json  (2 KB)
    models/thresholds_by_budget.json  (2 KB)
    preprocessing/btc_preprocessing.json  (0 KB)
    preprocessing/eth_preprocessing.json  (0 KB)
    protocol/xgboost_protocol.json  (4 KB)
    reports/evaluation_report.json  (82 KB)
    reports/limitations.md  (1 KB)
    reports/lofo_report.json  (7 KB)
    reports/selection_table.json  (3 KB)
    reports/shap_report.json  (3 KB)
    reports/training_report.json  (17 KB)
    reports/validation_results.json  (43 KB)
    scores/btc_xgb_final_scores_l1.parquet  (2676 KB)
    scores/btc_xgb_final_scores_l3.parquet  (2677 KB)
    scores/eth_xgb_final_scores_l1.parquet  (2678 KB)
    scores/eth_xgb_fina

## 10 · Save and declare the final set

In [ ]:
A3_TEXT = r"""# Protocol B2 – Amendment A3 (decision rules of notebook 31) – 2026-10-14
Written BEFORE the CAL measurement of XGBoost v2 and Transformer v2 exists (their CAL scores are saved by notebooks 29/30 but have not been analysed). For AE-ctx the genuine CAL alarm rates were already seen in notebook 28 (its adoption of N1 is frozen, not changed). TEST-B / TEST-3 are untouched.

1. **N0/N1 adoption per model family (XGBoost v2, Transformer v2).** N1 is adopted iff H-B1 AND H-B2, exactly as in the protocol. H-B1: genuine CAL, primary budget 2/day, inflation = mean over BTC/ETH of realised episodes/day divided by 2; N0 with static threshold T0 versus N1 with rolling threshold T1; N1 must cut the excess over the budget by at least 50% and stay within [0.5, 2] x budget. H-B2: VAL-3 recall at the primary budget, macro over all 8 families, mean BTC/ETH, N1 not more than 0.05 below N0. **Rule used for the H-B2 recall: T1c** (amendment A1/A2 reasoning: T1 on the dense synthetic VAL-3 is depressed). T1 (pre-registered) and T0 values are printed next to it; if the T1 reading would give a different adoption this is reported openly, and the T1c reading is the one used.
2. **Deployed threshold rule of every final model: T1** (rolling 7-day own-score quantile, q per budget frozen from the genuine DEV bars). N0/N1 changes only the normalisation. T0 is a drift stress test, never a deployed rule.
3. **CAL-3 recall** is reported under T1c (primary), T1 and T0, at budgets 1/2/4, per family, with event-bootstrap 95% intervals and the chance baseline (genuine alarm series circularly shifted). CAL-3 events lie inside bars 1440..44639 (2024-08-02..08-31); only bars < 44640 of the CAL series are used; thresholds and q come from DEV only.
4. **Union.** The union of the three final models is reported at budgets 1/2/4 per model with the realised genuine union episodes/day; nothing is selected on CAL from the union.
5. **v1 comparison and H-B6.** The v1 model files are not re-scored in notebook 31 (their file formats are listed in the notebook for notebook 31b). H-B6 is decided on TEST-3 in notebook 32 (T1c, T1 and T0 shown side by side; the T1 realised-burden condition as pre-registered).
6. **Aug 5 2024 (market crash day)** is a descriptive stress check (episodes that day versus other CAL days); no decision is attached.
7. After notebook 31 saves its outputs, `declared_final_set_v1.json` is the frozen final set; no model, normalisation, threshold rule or q is changed before TEST.
"""

In [ ]:
RATES.to_csv(OUT/'cal_genuine_rates.csv', index=False); DAYS.to_csv(OUT/'cal_genuine_days.csv', index=False); CAL3.to_csv(OUT/'cal3_recall.csv', index=False); FAMT.to_csv(OUT/'cal3_by_family.csv', index=False)
UNI.to_csv(OUT/'cal3_union.csv', index=False); COMP.to_csv(OUT/'cal3_complementarity.csv', index=False); CRASHT.to_csv(OUT/'aug5_crash_check.csv', index=False)
(OUT/'cal_decisions_v1.json').write_text(json.dumps(DEC, indent=1)); a3p = OUT/'protocol_b2_amendment_A3.md'; a3p.write_text(A3_TEXT)
sel = {'AE-ctx fmax': 'AE-ctx v2 (conditional AE, bottleneck 3), score = per-bar fmax', 'XGBoost v2': {m_: XD['selected'][m_] for m_ in MODES}, 'Transformer v2': {m_: TD['final'][m_] for m_ in MODES}}
DECL = {'status': 'DECLARED before TEST-B / TEST-3 were opened. No model, normalisation, threshold rule or q may change before notebook 32.',
        'declared_utc': datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z', 'protocol_sha256': ph, 'amendment_A3_sha256': hashlib.sha256(A3_TEXT.encode()).hexdigest(), 'lib_sha256_16': sha_file('b2_lib.py'),
        'primary_budget_eps_per_day': PRIM, 'deployed_threshold_rule': 'T1 (rolling 7-day own-score quantile, q per budget frozen from genuine DEV bars)', 'models': {}}
for model in MODELS:
    mode = FINALN[model]
    DECL['models'][model] = {'normalisation': mode, 'configuration': sel[model] if model == 'AE-ctx fmax' else sel[model][mode], 'seeds': [11, 22, 33] if model != 'AE-ctx fmax' else 'see notebook 28',
        'q_T1_by_budget': {str(b_): {s: par(model, s, mode, b_)['T1'] for s in SYMBOLS} for b_ in BUD}, 'T0_threshold_by_budget (drift stress test only)': {str(b_): {s: par(model, s, mode, b_)['T0'] for s in SYMBOLS} for b_ in BUD},
        'score_files_sha256_16': {s: [f'scores/{FILES[model]}_{s}_{mode}.npz', sha_file(OUT/'scores'/f'{FILES[model]}_{s}_{mode}.npz')] for s in SYMBOLS}}
DECL['union_rule'] = 'each model alarms at its own T1 threshold; union genuine burden measured on CAL (see cal3_union.csv); fusion (Integrity Score) is a separate protocol'
(OUT/'declared_final_set_v1.json').write_text(json.dumps(DECL, indent=1)); say('saved CSVs, cal_decisions_v1.json, protocol_b2_amendment_A3.md and declared_final_set_v1.json'); say(json.dumps({m_: DECL['models'][m_]['normalisation'] for m_ in MODELS}))

saved CSVs, cal_decisions_v1.json, protocol_b2_amendment_A3.md and declared_final_set_v1.json
{"AE-ctx fmax": "N1", "XGBoost v2": "N0", "Transformer v2": "N1"}


## 11 · Share block

In [ ]:
block = '\n'.join(LINES); (OUT/'b2_step5_share_block.txt').write_text(block)
print('\n===== SHARE BLOCK (copy everything below, or attach models_b2_v1/b2_step5_share_block.txt) =====\n' + block[-14000:])


===== SHARE BLOCK (copy everything below, or attach models_b2_v1/b2_step5_share_block.txt) =====
        0.06            0.06            0.32            0.80            0.14            0.08
   AE-ctx fmax   N0   T1            0.04            0.04            0.02            0.02            0.14            0.64            0.02            0.00
   AE-ctx fmax   N0   T0            0.12            0.04            0.06            0.06            0.24            0.70            0.10            0.04
   AE-ctx fmax   N1  T1c            0.08            0.12            0.08            0.12            0.22            0.70            0.20            0.12
   AE-ctx fmax   N1   T1            0.04            0.08            0.06            0.08            0.08            0.50            0.06            0.02
   AE-ctx fmax   N1   T0            0.06            0.08            0.06            0.08            0.12            0.56            0.10            0.04
    XGBoost v2   N0  T1c            0.68    